# Setup metadata FANTOM5 time course data
## Setup metadata to download CTSS bed files (hg19)
### Author: Martin Loza
### Date: 25/12/19

We need to download the data from FANTOM5, then, we want to create a metadata to later use for sample labeling

In [78]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
})

# Local variables 
seed = 777
date = "251219"

# Define colors for strand plots
red = "#E41A1C"
blue = "#090a0bff"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

fantom_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/FANTOM5/"
out_dir = fantom_dir

# Local Functions


### Load and setup FANTOM5 metadata

In [79]:
#Load the metadata for FANTOM5 CAGE libraries
md_file = paste0(fantom_dir,"human.timcourse.hCAGE_251219.tsv")
metadata = read.table(md_file, sep = "\t", header = TRUE, 
                        comment.char = "", fill = TRUE, row.names = NULL)
dim(metadata)
head(metadata,2)

[1] 783  28

,Extract.Name,Comment..rna_tube.,Comment..sample_name.,Comment..organism.,Protocol.REF,Parameter..rna_extraction.,Extract.Name.1,Comment..rna_id.,Comment..Material.Type.,Comment..ratio.260.280.,⋯,Protocol.REF.2,Parameter..sequence_protocol.,Parameter..machine_name.,Parameter..run_name.,Parameter..flowcell.channel.,File.Name,Comment..sequence_raw_file.,Protocol.REF.3,Parameter..sex.,File.Name.1
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,13305-142I2,13305-142I2,"Monocyte-derived macrophages response to udorn influenza infection, 00hr00min, donor1 (868_121:Ud_0h)",Homo sapiens,standard whole cell,standard whole cell,13305-142I2.rna,13305-142I2.rna,total RNA,"1.99,2.16,1.9",⋯,OP-HELISCOPE-sequencing-v1.0,OP-HELISCOPE-sequencing-v1.0,NA,NA,fc1.ch03,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,delve_align,UNDEFINED_SEX_TYPE,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.nobarcode.bam
2,13306-142I3,13306-142I3,"Monocyte-derived macrophages response to udorn influenza infection, 02hr00min, donor1 (868_121:Ud_2h)",Homo sapiens,standard whole cell,standard whole cell,13306-142I3.rna,13306-142I3.rna,total RNA,"2.14,2.19,2.11",⋯,OP-HELISCOPE-sequencing-v1.0,OP-HELISCOPE-sequencing-v1.0,NA,NA,fc2.ch22,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,delve_align,UNDEFINED_SEX_TYPE,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.nobarcode.bam


In [80]:
colnames(metadata)  
metadata[1:2, 20:28]

[1] "Extract.Name"                   "Comment..rna_tube."            
 [3] "Comment..sample_name."          "Comment..organism."            
 [5] "Protocol.REF"                   "Parameter..rna_extraction."    
 [7] "Extract.Name.1"                 "Comment..rna_id."              
 [9] "Comment..Material.Type."        "Comment..ratio.260.280."       
[11] "Comment..ratio.260.230."        "Comment..concentration."       
[13] "Comment..RNA.integrity.number." "Protocol.REF.1"                
[15] "Comment..lsid."                 "Parameter..library_protocol."  
[17] "Library.Name"                   "Comment..library_id."          
[19] "Protocol.REF.2"                 "Parameter..sequence_protocol." 
[21] "Parameter..machine_name."       "Parameter..run_name."          
[23] "Parameter..flowcell.channel."   "File.Name"                     
[25] "Comment..sequence_raw_file."    "Protocol.REF.3"                
[27] "Parameter..sex."                "File.Name.1"

,Parameter..sequence_protocol.,Parameter..machine_name.,Parameter..run_name.,Parameter..flowcell.channel.,File.Name,Comment..sequence_raw_file.,Protocol.REF.3,Parameter..sex.,File.Name.1
,<chr>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,OP-HELISCOPE-sequencing-v1.0,NA,NA,fc1.ch03,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,delve_align,UNDEFINED_SEX_TYPE,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.nobarcode.bam
2,OP-HELISCOPE-sequencing-v1.0,NA,NA,fc2.ch22,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,/sequencedata/heliscope/srf/2011_05_31_R4_30Q_1100FOV/,delve_align,UNDEFINED_SEX_TYPE,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.nobarcode.bam


In [81]:
# select columns of interest
sel_cols <- c('Comment..sample_name.','Comment..organism.','Protocol.REF','Comment..Material.Type.','File.Name.1')
metadata_sel <- metadata %>%
    select(all_of(sel_cols))
head(metadata_sel,2)

,Comment..sample_name.,Comment..organism.,Protocol.REF,Comment..Material.Type.,File.Name.1
,<chr>,<chr>,<chr>,<chr>,<chr>
1,"Monocyte-derived macrophages response to udorn influenza infection, 00hr00min, donor1 (868_121:Ud_0h)",Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.nobarcode.bam
2,"Monocyte-derived macrophages response to udorn influenza infection, 02hr00min, donor1 (868_121:Ud_2h)",Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.nobarcode.bam


In [82]:
# rename columns for easier handling
colnames(metadata_sel) <- c('sample_name','organism','protocol_ref','material_type','file_name')
head(metadata_sel,3)

,sample_name,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>
1,"Monocyte-derived macrophages response to udorn influenza infection, 00hr00min, donor1 (868_121:Ud_0h)",Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.nobarcode.bam
2,"Monocyte-derived macrophages response to udorn influenza infection, 02hr00min, donor1 (868_121:Ud_2h)",Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.nobarcode.bam
3,"Monocyte-derived macrophages response to udorn influenza infection, 07hr00min, donor1 (868_121:Ud_7h)",Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2007hr00min%2c%20donor1%20%28868_121%3aUd_7h%29.CNhs13556.13307-142I4.hg19.nobarcode.bam


Let's recover the time and the donor/replicate. They are inside the sample_name as : data_name, time, donor_replicate (extra)

In [83]:
# Let's recover the data_name, time point and replicate information from the sample_name column
# The sample_name column has the following format: data_name , time_point, donor_replicate (extra information in parentheses)
metadata_sel <- metadata_sel %>%
    mutate(
        data_name = str_trim(str_extract(sample_name, "^[^,]+")),
        time_point = str_trim(str_extract(sample_name, "(?<=,)[^,]+(?=,)")),
        donor_replicate = str_trim(str_extract(sample_name, "(?<=,)[^,]+$"))
    ) %>%
    select(-sample_name) %>%
    select( data_name, time_point, donor_replicate, everything())
head(metadata_sel,3)

,data_name,time_point,donor_replicate,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Monocyte-derived macrophages response to udorn influenza infection,00hr00min,donor1 (868_121:Ud_0h),Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.nobarcode.bam
2,Monocyte-derived macrophages response to udorn influenza infection,02hr00min,donor1 (868_121:Ud_2h),Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.nobarcode.bam
3,Monocyte-derived macrophages response to udorn influenza infection,07hr00min,donor1 (868_121:Ud_7h),Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2007hr00min%2c%20donor1%20%28868_121%3aUd_7h%29.CNhs13556.13307-142I4.hg19.nobarcode.bam


Let's remove the extra information in the parenthesis

In [84]:
# Remove the parantheses information in the donor_replicate column
metadata_sel <- metadata_sel %>%
    mutate(
        donor_replicate = str_trim(str_replace(donor_replicate, "\\s*\\(.*\\)$", ""))
    )
head(metadata_sel,3)

,data_name,time_point,donor_replicate,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Monocyte-derived macrophages response to udorn influenza infection,00hr00min,donor1,Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.nobarcode.bam
2,Monocyte-derived macrophages response to udorn influenza infection,02hr00min,donor1,Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.nobarcode.bam
3,Monocyte-derived macrophages response to udorn influenza infection,07hr00min,donor1,Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2007hr00min%2c%20donor1%20%28868_121%3aUd_7h%29.CNhs13556.13307-142I4.hg19.nobarcode.bam


In [85]:
cat("Any duplicated file names found in the metadata: \n", any(duplicated(metadata_sel$file_name))) 

Any duplicated file names found in the metadata: 
 FALSE

Now, to download the actuan CTSS bed files, we need to change the last part of the file_name. We would like to replace .nobarcode.bam for .ctss.bed.gz

In [86]:
# Replace the extension of the file names to .ctss.bed.gz
metadata_sel <- metadata_sel %>%
    mutate(
        file_name = str_replace(file_name, "\\.nobarcode.bam$", ".ctss.bed.gz")
    )

head(metadata_sel,3)

,data_name,time_point,donor_replicate,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Monocyte-derived macrophages response to udorn influenza infection,00hr00min,donor1,Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.ctss.bed.gz
2,Monocyte-derived macrophages response to udorn influenza infection,02hr00min,donor1,Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.ctss.bed.gz
3,Monocyte-derived macrophages response to udorn influenza infection,07hr00min,donor1,Homo sapiens,standard whole cell,total RNA,Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2007hr00min%2c%20donor1%20%28868_121%3aUd_7h%29.CNhs13556.13307-142I4.hg19.ctss.bed.gz


Let's add the website for downloading with proper URL encoding

Note: FANTOM5 server requires double-encoded URLs for special characters like spaces, commas, and parentheses to work properly with wget/curl downloads.

In [87]:
# add the website link to the file names with proper URL encoding
metadata_sel <- metadata_sel %>%
    mutate(
        # First ensure we only have the filename part (remove any existing base URL)
        file_name = ifelse(
            str_starts(file_name, "https://"),
            str_extract(file_name, "[^/]+\\.ctss\\.bed\\.gz$|[^/]+\\.nobarcode\\.bam$"),
            file_name
        ),
        # Then create the full URL path
        file_name = paste0("https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/", file_name),
        # Finally apply proper URL encoding for FANTOM5 server
        # The server expects double-encoded URLs for special characters
        file_name = str_replace_all(file_name, " ", "%2520"),  # space to double-encoded space
        file_name = str_replace_all(file_name, ",", "%252c"),  # comma to double-encoded comma  
        file_name = str_replace_all(file_name, "\\(", "%2528"), # ( to double-encoded (
        file_name = str_replace_all(file_name, "\\)", "%2529")  # ) to double-encoded )
    )
head(metadata_sel,3)

,data_name,time_point,donor_replicate,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Monocyte-derived macrophages response to udorn influenza infection,00hr00min,donor1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2000hr00min%2c%20donor1%20%28868_121%3aUd_0h%29.CNhs13554.13305-142I2.hg19.ctss.bed.gz
2,Monocyte-derived macrophages response to udorn influenza infection,02hr00min,donor1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2002hr00min%2c%20donor1%20%28868_121%3aUd_2h%29.CNhs13555.13306-142I3.hg19.ctss.bed.gz
3,Monocyte-derived macrophages response to udorn influenza infection,07hr00min,donor1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Monocyte-derived%20macrophages%20response%20to%20udorn%20influenza%20infection%2c%2007hr00min%2c%20donor1%20%28868_121%3aUd_7h%29.CNhs13556.13307-142I4.hg19.ctss.bed.gz


Let's make sure we don't have any .bam files

In [88]:
metadata_sel <- metadata_sel %>% filter(!str_detect(file_name, "bam"))

Let'select some files of interest

In [89]:
unique(metadata_sel$data_name)
length(unique(metadata_sel$data_name))

[1] "Monocyte-derived macrophages response to udorn influenza infection"            
 [2] "Lymphatic Endothelial cells response to VEGFC"                                 
 [3] "Monocyte-derived macrophages response to LPS"                                  
 [4] "Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification"
 [5] "Adipocyte differentiation"                                                     
 [6] "MCF7 breast cancer cell line response to HRG"                                  
 [7] "K562 erythroblastic leukemia response to hemin"                                
 [8] "MCF7 breast cancer cell line response to EGF1"                                 
 [9] "H9 Embryoid body cells"                                                        
[10] "H9 Embryonic Stem cells"                                                       
[11] "Melanocyte"                                                                    
[12] "mesenchymal stem cells (adipose derived)"                                      
[13] "Aortic smooth muscle cell response to FGF2"                                    
[14] "Aortic smooth muscle cell response to IL1b"                                    
[15] "Monocyte-derived macrophages response to mock influenza infection"             
[16] "HES3-GFP Embryonic Stem cells"                                                 
[17] "iPS differentiation to neuron"                                                 
[18] "Myoblast differentiation to myotubes"                                          
[19] "H1 embryonic stem cells differentiation to CD34+ HSC"                          
[20] "Saos-2 osteosarcoma cell line"                                                 
[21] "hIPS"                                                                          
[22] "hIPS +CCl2"                                                                    
[23] "293SLAM rinderpest infection"                                                  
[24] "COBL-a rinderpest infection"                                                   
[25] "COBL-a rinderpest(-C) infection"                                               
[26] "ARPE-19 EMT induced with TGF-beta and TNF-alpha"

[1] 26

In [90]:
# selected data series for the analysis
data_series_sel <- c(
    'Lymphatic Endothelial cells response to VEGFC',
    'Monocyte-derived macrophages response to LPS',
    'Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification',
    'MCF7 breast cancer cell line response to EGF1',
    'Aortic smooth muscle cell response to FGF2',
    'mesenchymal stem cells (adipose derived)',
    'iPS differ entiation to neuron'
)

cat("Number of rows in the metadata before filtering: ", nrow(metadata_sel), "\n")
# Filter the metadata to keep only the selected data series
metadata_sel <- metadata_sel %>%
    filter(data_name %in% data_series_sel)
cat("Number of rows in the metadata after filtering: ", nrow(metadata_sel), "\n")
head(metadata_sel,3)

Number of rows in the metadata before filtering:  761 
Number of rows in the metadata after filtering:  298 


,data_name,time_point,donor_replicate,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Lymphatic Endothelial cells response to VEGFC,00hr00min,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Lymphatic%20Endothelial%20cells%20response%20to%20VEGFC%2c%2000hr00min%2c%20biol_rep1%20%28MM%20XIX%20-%201%29.CNhs11936.12260-130A1.hg19.ctss.bed.gz
2,Lymphatic Endothelial cells response to VEGFC,08hr,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Lymphatic%20Endothelial%20cells%20response%20to%20VEGFC%2c%2008hr%2c%20biol_rep1%20%28MM%20XIX%20-%2016%29.CNhs11937.12275-130B7.hg19.ctss.bed.gz
3,Monocyte-derived macrophages response to LPS,00hr00min,donor1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Monocyte-derived%20macrophages%20response%20to%20LPS%2c%2000hr00min%2c%20donor1%20%28t1%20Subject1%29.CNhs11941.12698-135D7.hg19.ctss.bed.gz


Now, let's keep only one replicate or donor. In this case the one with the largest number of samples

In [91]:
# Keep one donor/replicate with the largest number of time points per data series
selected_donors <- metadata_sel %>%
    group_by(data_name, donor_replicate) %>%
    mutate(
        n_time_points = n()
    ) %>%
    ungroup() %>%
    group_by(data_name) %>%
    arrange(donor_replicate, desc(n_time_points)) %>%
    filter(n_time_points == max(n_time_points)) %>%
    slice(1) %>%  # in case of ties, keep the first one
    ungroup() %>%
    select(data_name, donor_replicate, n_time_points)

selected_donors
sum(selected_donors$n_time_points)

data_name,donor_replicate,n_time_points
<chr>,<chr>,<int>
Aortic smooth muscle cell response to FGF2,biol_rep2,10
Lymphatic Endothelial cells response to VEGFC,biol_rep1,16
MCF7 breast cancer cell line response to EGF1,biol_rep2,16
Monocyte-derived macrophages response to LPS,donor2,25
Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification,biol_rep1,18
mesenchymal stem cells (adipose derived),biol_rep2,18


[1] 103

In [92]:
# Keep only information from the selected donors
selected_donor_ids <- selected_donors %>% mutate(id = paste0(data_name,"___",donor_replicate)) %>% select(id) 

metadata_sel <- metadata_sel %>%
    mutate(id = paste0(data_name,"___",donor_replicate)) %>%
    filter(id %in% selected_donor_ids$id) %>%
    select(-id)

cat("Number of rows in the metadata after keeping only selected donors: ", nrow(metadata_sel), "\n")

Number of rows in the metadata after keeping only selected donors:  103 


In [93]:
head(metadata_sel)

,data_name,time_point,donor_replicate,organism,protocol_ref,material_type,file_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Lymphatic Endothelial cells response to VEGFC,00hr00min,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Lymphatic%20Endothelial%20cells%20response%20to%20VEGFC%2c%2000hr00min%2c%20biol_rep1%20%28MM%20XIX%20-%201%29.CNhs11936.12260-130A1.hg19.ctss.bed.gz
2,Lymphatic Endothelial cells response to VEGFC,08hr,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Lymphatic%20Endothelial%20cells%20response%20to%20VEGFC%2c%2008hr%2c%20biol_rep1%20%28MM%20XIX%20-%2016%29.CNhs11937.12275-130B7.hg19.ctss.bed.gz
3,Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification,00hr15min,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Saos-2%20osteosarcoma%20treated%20with%20ascorbic%20acid%20and%20BGP%20to%20induce%20calcification%2c%2000hr15min%2c%20biol_rep1%20%28A1%20T1%29.CNhs12381.12663-134I8.hg19.ctss.bed.gz
4,Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification,00hr30min,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Saos-2%20osteosarcoma%20treated%20with%20ascorbic%20acid%20and%20BGP%20to%20induce%20calcification%2c%2000hr30min%2c%20biol_rep1%20%28A1%20T2%29.CNhs12382.12664-134I9.hg19.ctss.bed.gz
5,Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification,00hr45min,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Saos-2%20osteosarcoma%20treated%20with%20ascorbic%20acid%20and%20BGP%20to%20induce%20calcification%2c%2000hr45min%2c%20biol_rep1%20%28A1%20T3%29.CNhs12383.12665-135A1.hg19.ctss.bed.gz
6,Saos-2 osteosarcoma treated with ascorbic acid and BGP to induce calcification,01hr00min,biol_rep1,Homo sapiens,standard whole cell,total RNA,https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Saos-2%20osteosarcoma%20treated%20with%20ascorbic%20acid%20and%20BGP%20to%20induce%20calcification%2c%2001hr00min%2c%20biol_rep1%20%28A1%20T4%29.CNhs12384.12666-135A2.hg19.ctss.bed.gz


Let's now to standardize the time points for comparison

In [94]:
# Let's examine the different time point formats
cat("Unique time points in the data:\n")
unique_time_points <- sort(unique(metadata_sel$time_point))
print(unique_time_points)
cat("\nNumber of unique time points:", length(unique_time_points), "\n")

Unique time points in the data:
 [1] "00hr00min"                "00hr15min"               
 [3] "00hr30min"                "00hr45min"               
 [5] "01hr"                     "01hr00min"               
 [7] "01hr20min"                "01hr40min"               
 [9] "02hr"                     "02hr00min"               
[11] "02hr30min"                "03hr"                    
[13] "03hr00min"                "03hr30min"               
[15] "04hr"                     "05hr"                    
[17] "06hr"                     "07hr"                    
[19] "08hr"                     "10hr"                    
[21] "12hr"                     "14hr"                    
[23] "16hr"                     "18hr"                    
[25] "20hr"                     "22hr"                    
[27] "24hr"                     "36hr"                    
[29] "48hr"                     "adipogenic induction"    
[31] "day04"                    "day07"                   
[33] "day14"            


Number of unique time points: 36 


We want to create a simple time point and remove those with not typical time points, like undifferentiated control

In [95]:
# Remove samples with non-standard time points
non_standard_times <- c("undifferentiated control", "adipogenic induction")

cat("Removing samples with non-standard time points:\n")
cat("Before filtering:", nrow(metadata_sel), "samples\n")

metadata_sel <- metadata_sel %>%
    filter(!time_point %in% non_standard_times)

cat("After filtering:", nrow(metadata_sel), "samples\n")

# Create simple sequential time point numbering for each data series
metadata_sel <- metadata_sel %>%
    group_by(data_name) %>%
    arrange(time_point) %>%  # Sort by original time_point first
    mutate(
        time_sequence = row_number()  # Create simple 1, 2, 3, ... numbering
    ) %>%
    ungroup()


Removing samples with non-standard time points:
Before filtering: 103 samples
After filtering: 85 samples


In [96]:
# head(metadata_sel %>% select(data_name, time_point, time_sequence) %>% filter(data_name == "Lymphatic Endothelial cells response to VEGFC"), 15)

In [97]:
# arrange by data_name and time_sequence
metadata_sel <- metadata_sel %>%
    arrange(data_name, time_sequence)

In [98]:
# save the metadata for future use
out_metadata_file = paste0(out_dir, "FANTOM5_timecourse_metadata_selected_series_", date, ".tsv")
write.table(metadata_sel, file = out_metadata_file, sep = "\t", quote = FALSE, row.names = FALSE)

In [99]:
# save also for supplementary table
out_sup_table = paste0("/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Supplementary/")
write.table(metadata_sel, file = paste0(out_sup_table, "Supplementary_Table_FANTOM5_timecourse_metadata_selected_series_", date, ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

### Create URL list file for easy downloading

A simple text file with one URL per line is much easier for bash scripts

In [100]:
# Create a simple text file with URLs for easy bash downloading
url_list_file = paste0(out_dir, "FANTOM5_download_urls_", date, ".txt")

# Extract URLs and apply proper encoding
url_list <- metadata_sel %>%
    # Start fresh from the filenames to avoid double base URL issue
    mutate(
        # Get clean filename (extract just the filename part)
        clean_filename = str_replace(file_name, ".*basic/human\\.timecourse\\.hCAGE/", ""),
        # Build properly encoded URLs - need double encoding for FANTOM5 server
        encoded_url = paste0("https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/", clean_filename),
        # Apply double encoding: %20 -> %2520, etc.
        encoded_url = str_replace_all(encoded_url, "%20", "%2520"),
        encoded_url = str_replace_all(encoded_url, "%2c", "%252c"),
        encoded_url = str_replace_all(encoded_url, "%28", "%2528"),
        encoded_url = str_replace_all(encoded_url, "%29", "%2529")
    ) %>%
    select(encoded_url)

# Save URL list
writeLines(url_list$encoded_url, url_list_file)

cat("Created URL list file with", nrow(url_list), "URLs\n")
cat("File:", url_list_file, "\n")

# Show first few URLs
cat("\nFirst 3 URLs:\n")
head(url_list$encoded_url, 3)

Created URL list file with 85 URLs
File: /mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/FANTOM5/FANTOM5_download_urls_251219.txt 

First 3 URLs:


[1] "https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Aortic%2520smooth%2520muscle%2520cell%2520response%2520to%2520FGF2%252c%252000hr00min%252c%2520biol_rep2%2520%2528LK2%2529.CNhs13358.12740-135I4.hg19.ctss.bed.gz"
[2] "https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Aortic%2520smooth%2520muscle%2520cell%2520response%2520to%2520FGF2%252c%252000hr15min%252c%2520biol_rep2%2520%2528LK5%2529.CNhs13359.12741-135I5.hg19.ctss.bed.gz"
[3] "https://fantom.gsc.riken.jp/5/datafiles/latest/basic/human.timecourse.hCAGE/Aortic%2520smooth%2520muscle%2520cell%2520response%2520to%2520FGF2%252c%252000hr30min%252c%2520biol_rep2%2520%2528LK8%2529.CNhs13360.12742-135I6.hg19.ctss.bed.gz"